# mIF Pipeline Stage Notebook: Final SpatialData Assembly

Use this notebook in the `harpy` environment.

It assumes the earlier artifact-producing stages have already completed in the `instanseg_nimbus` environment.

This notebook covers:
- inspecting the resolved artifact paths
- running `assemble_spatialdata(...)`
- inspecting the resulting `SpatialData` object


In [1]:
from pathlib import Path
import json
import platform
import traceback

import mif_pipeline
from mif_pipeline import assemble_spatialdata, load_config
from mif_pipeline.config import get_slide_config
import mif_pipeline.spatialdata_builder as spatialdata_builder_module

print("python:", platform.python_version())
print("mif_pipeline:", getattr(mif_pipeline, "__file__", "unknown"))


python: 3.12.13
mif_pipeline: /home/ratnayn/codex/mIF-pipeline/src/mif_pipeline/__init__.py


In [2]:
# Edit these for your current run.
CONFIG_PATH = Path("~/codex/mIF-pipeline/prototyping/prototype_v2-Crop.yaml").expanduser()
SLIDE_ID = "SLIDE-0329_crop_2048"

# Optional Dask cluster for Harpy/SpatialData assembly.
USE_DASK_CLUSTER = False
DASK_N_WORKERS = 4
DASK_THREADS_PER_WORKER = 1
DASK_MEMORY_LIMIT = "16GB"

FORCE = True
RUN_ASSEMBLE = True
ASSEMBLE_DRY_RUN = not RUN_ASSEMBLE


In [3]:
def show(value):
    print(json.dumps(value, indent=2, default=str))


def stage_call(label, func, *args, **kwargs):
    print(f"\n=== {label} ===")
    try:
        result = func(*args, **kwargs)
        show(result)
        return result
    except Exception as exc:
        print(f"{type(exc).__name__}: {exc}")
        traceback.print_exc()
        return None


In [4]:
print("config path:", CONFIG_PATH)
print("config exists:", CONFIG_PATH.exists())

config = load_config(CONFIG_PATH)
raw_slide = config["slides"][SLIDE_ID]
top_level_nimbus = config.get("nimbus") or {}
multislide_block = top_level_nimbus.get("multislide") or {}
multislide_enabled = bool(multislide_block.get("enabled", False))
runtime_slide_ids = list(multislide_block.get("slide_ids") or [SLIDE_ID]) if multislide_enabled else [SLIDE_ID]

if SLIDE_ID not in runtime_slide_ids:
    raise ValueError(f"SLIDE_ID {SLIDE_ID!r} is not in the resolved runtime slide list: {runtime_slide_ids}")

runtime_slides = {target_slide_id: get_slide_config(config, target_slide_id) for target_slide_id in runtime_slide_ids}
artifact_paths = spatialdata_builder_module._spatialdata_paths(runtime_slides[SLIDE_ID])
runtime_artifact_paths = {
    target_slide_id: spatialdata_builder_module._spatialdata_paths(target_slide)
    for target_slide_id, target_slide in runtime_slides.items()
}
assemble_targets = list(runtime_slide_ids)

print("slide_id:", SLIDE_ID)
print("nimbus_multislide_enabled:", multislide_enabled)
print("runtime_slide_ids:", runtime_slide_ids)
print("assemble_targets:", assemble_targets)
show({
    "top_level_nimbus": top_level_nimbus,
    "slide_block": raw_slide,
    "artifact_paths": {key: str(value) for key, value in artifact_paths.items()},
    "runtime_artifact_paths": {slide_id: {key: str(value) for key, value in paths.items()} for slide_id, paths in runtime_artifact_paths.items()},
})


config path: /home/ratnayn/codex/mIF-pipeline/prototyping/prototype_v2-Crop.yaml
config exists: True
slide_id: SLIDE-0329_crop_2048
nimbus_multislide_enabled: True
runtime_slide_ids: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2']
assemble_targets: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2']
{
  "top_level_nimbus": {
    "enabled": true,
    "output_dir": "nimbus",
    "channels": [
      "R1_DAPI",
      "R4_GFP_POLY_AF488",
      "R4_P19_POLYRAT",
      "R4_ANXA10_647"
    ],
    "channel_chunk_size": 1,
    "join_keys": [
      "fov",
      "cell_id"
    ],
    "batch_size": 16,
    "save_predictions": true,
    "compile_model": false,
    "quantile": 0.999,
    "n_subset": 50,
    "clip_values": [
      0,
      2
    ],
    "multiprocessing": true,
    "multislide": {
      "enabled": true,
      "output_dir": "/mnt/c/analysis/Data_prototype/crop_nimbus_multislide",
      "per_slide_output_dirname": "per_slide",
      "slide_ids": [
        "SLIDE-0329_crop_2048",
      

## Expected Input Artifacts

These are the files this notebook expects to already exist from the earlier stage notebook.

If `nimbus.multislide.enabled: true`, the resolved `nimbus_table_path` below should point to the per-slide table under the multislide output root rather than the slide-local Nimbus folder.

If multislide is enabled in the YAML, this notebook will assemble SpatialData for every slide in the resolved runtime list automatically.


In [5]:
for target_slide_id, target_paths in runtime_artifact_paths.items():
    print(f"\n=== expected artifacts: {target_slide_id} ===")
    for key, value in target_paths.items():
        path = Path(value)
        print(f"{key}: {path}")
        print(f"  exists: {path.exists()}")

nimbus_multislide_enabled = bool((((config.get("nimbus") or {}).get("multislide") or {}).get("enabled", False)))
print("nimbus_multislide_enabled:", nimbus_multislide_enabled)



=== expected artifacts: SLIDE-0329_crop_2048 ===
full_merge_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif
  exists: True
cell_mask_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff
  exists: True
nuclear_mask_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_nuclear.tiff
  exists: True
nimbus_table_path: /mnt/c/analysis/Data_prototype/crop_nimbus_multislide/per_slide/SLIDE-0329_crop_2048/cell_table_full.csv
  exists: True
store_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr
  exists: True

=== expected artifacts: SLIDE-0329_crop_2048_2 ===
full_merge_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/SLIDE-0329_crop_2048_2_full.ome.tif
  exists: True
cell_mask_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_c

## Final SpatialData Assembly

This stage imports the merged image with the `tiffslide` loader, imports the cell and nuclear masks as labels, optionally imports the Nimbus table, runs raster aggregation, and writes the canonical final `SpatialData` store.

When multislide Nimbus is enabled in the YAML, `assemble_spatialdata(...)` will automatically read the per-slide Nimbus table path resolved above.

If multislide is enabled in the YAML, we call `assemble_spatialdata(...)` once per runtime slide automatically.


In [6]:
assemble_results = {}
for target_slide_id in assemble_targets:
    assemble_results[target_slide_id] = stage_call(
        f"assemble_spatialdata({target_slide_id}, dry_run={ASSEMBLE_DRY_RUN})",
        assemble_spatialdata,
        config,
        target_slide_id,
        force=FORCE,
        dry_run=ASSEMBLE_DRY_RUN,
        return_sdata=not ASSEMBLE_DRY_RUN,
    )



=== assemble_spatialdata(SLIDE-0329_crop_2048, dry_run=False) ===
[spatialdata] loading image pyramid with tiffslide/zarr: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif
[spatialdata] image_load_seconds completed in 0.13s
[spatialdata] loading label masks for SLIDE-0329_crop_2048
[spatialdata] mask_load_seconds completed in 0.08s
[spatialdata] deriving shapes from labels: ['cell_labels', 'nuclear_labels']
[spatialdata] using Dask scheduler='processes' for CPU vectorization
[spatialdata] shape_derivation_seconds completed in 9.59s
[spatialdata] aggregating intensity tables for ['cell_labels', 'nuclear_labels']


2026-04-19 10:14:14.558 | WARNING  | harpy.utils._aggregate:_center_of_mass:884 - Count was computed for a different index. Recalculating.
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/harpy/table/_allocation_intensity.py:338: ImplicitModificationWarning: Setting element `.obsm['spatial']` of view, initializing view as actual.
  adata.obsm[spatial_key] = coordinates
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
2026-04-19 10:14:17.734 | WARNING  | harpy.utils._aggregate:_center_of_mass:884 - Count was computed for a different index. Recalculating.


[spatialdata] aggregation_seconds completed in 6.63s
[spatialdata] importing Nimbus table: /mnt/c/analysis/Data_prototype/crop_nimbus_multislide/per_slide/SLIDE-0329_crop_2048/cell_table_full.csv


/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/harpy/table/_allocation_intensity.py:338: ImplicitModificationWarning: Setting element `.obsm['spatial']` of view, initializing view as actual.
  adata.obsm[spatial_key] = coordinates
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


[spatialdata] nimbus_import_seconds completed in 0.05s


...

[spatialdata] aggregation_seconds completed in 6.28s
[spatialdata] importing Nimbus table: /mnt/c/analysis/Data_prototype/crop_nimbus_multislide/per_slide/SLIDE-0329_crop_2048_2/cell_table_full.csv


/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/harpy/table/_allocation_intensity.py:338: ImplicitModificationWarning: Setting element `.obsm['spatial']` of view, initializing view as actual.
  adata.obsm[spatial_key] = coordinates
/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


[spatialdata] nimbus_import_seconds completed in 0.05s


/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/spatialdata/models/models.py:1211: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


[spatialdata] writing final store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/SLIDE-0329_crop_2048_2_spatialdata.sdata.zarr
[spatialdata] write_seconds completed in 15.18s
{
  "slide_id": "SLIDE-0329_crop_2048_2",
  "status": "written",
  "dry_run": false,
  "image_loader": "tiffslide_zarr",
  "full_merge_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/SLIDE-0329_crop_2048_2_full.ome.tif",
  "store_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/SLIDE-0329_crop_2048_2_spatialdata.sdata.zarr",
  "cell_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/masks/SLIDE-0329_crop_2048_2_whole_cell.tiff",
  "nuclear_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/masks/SLIDE-0329_crop_2048_2_nuclear.tiff",
  "nimbus_table_path": "/mnt/c/analysis/Data_prototype/crop_nimbus_multislide/per_slide/SLIDE-0329_crop_2048_2/cell_table_full.csv",
 

In [23]:
import dask
from dask.base import get_scheduler
from dask.distributed import Client, LocalCluster, get_client

cluster = None
client = None

if USE_DASK_CLUSTER:
    cluster = LocalCluster(
        n_workers=DASK_N_WORKERS,
        threads_per_worker=DASK_THREADS_PER_WORKER,
        memory_limit=DASK_MEMORY_LIMIT,
        host="127.0.0.1",
    )
    client = Client(cluster)
    print("distributed client:", client)
    print("dashboard:", client.dashboard_link)
else:
    print("USE_DASK_CLUSTER = False")

try:
    print("dask config scheduler:", dask.config.get("scheduler", default=None))
except Exception as exc:
    print("dask config scheduler lookup failed:", exc)
print("current scheduler object:", get_scheduler())


/home/ratnayn/miniconda3/envs/harpy/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46677 instead
  warnings.warn(


distributed client: <Client: 'tcp://127.0.0.1:33025' processes=4 threads=4, memory=59.60 GiB>
dashboard: http://127.0.0.1:46677/status
dask config scheduler: dask.distributed
current scheduler object: <bound method Client.get of <Client: 'tcp://127.0.0.1:33025' processes=4 threads=4, memory=59.60 GiB>>


In [20]:
try:
    active_client = get_client()
    info = active_client.scheduler_info()
    print("distributed client active:", active_client)
    print("dashboard:", active_client.dashboard_link)
    print("workers:", len(info["workers"]))
    for address, worker in info["workers"].items():
        print(
            address,
            "threads=", worker.get("nthreads"),
            "memory_limit=", worker.get("memory_limit"),
        )
except ValueError:
    print("no distributed client active")


distributed client active: <Client: 'tcp://127.0.0.1:45159' processes=4 threads=4, memory=59.60 GiB>
dashboard: http://127.0.0.1:8787/status
workers: 4
tcp://127.0.0.1:36445 threads= 1 memory_limit= 16000000000
tcp://127.0.0.1:36673 threads= 1 memory_limit= 16000000000
tcp://127.0.0.1:37289 threads= 1 memory_limit= 16000000000
tcp://127.0.0.1:42405 threads= 1 memory_limit= 16000000000


In [21]:
if not ASSEMBLE_DRY_RUN:
    for target_slide_id, assemble_result in assemble_results.items():
        print(f"\n=== assembled object: {target_slide_id} ===")
        if assemble_result and "sdata" in assemble_result:
            sdata = assemble_result["sdata"]
            print("images:", list(sdata.images.keys()))
            print("labels:", list(sdata.labels.keys()))
            print("shapes:", list(sdata.shapes.keys()))
            print("tables:", list(sdata.tables.keys()))
        else:
            print("No in-memory SpatialData object available for this slide.")
else:
    print("No in-memory SpatialData object available. Set RUN_ASSEMBLE = True to inspect the assembled object.")



=== assembled object: SLIDE-0329_crop_2048 ===
images: ['full_image']
labels: ['cell_labels', 'nuclear_labels']
shapes: ['cell_boundaries', 'nuclear_boundaries']
tables: ['agg_cell_labels', 'agg_nuclear_labels', 'nimbus_table']

=== assembled object: SLIDE-0329_crop_2048_2 ===
images: ['full_image']
labels: ['cell_labels', 'nuclear_labels']
shapes: ['cell_boundaries', 'nuclear_boundaries']
tables: ['agg_cell_labels', 'agg_nuclear_labels', 'nimbus_table']


In [14]:
if not ASSEMBLE_DRY_RUN:
    for target_slide_id, assemble_result in assemble_results.items():
        if assemble_result:
            print(f"\n=== summaries: {target_slide_id} ===")
            print("aggregate tables:")
            for table_summary in assemble_result.get("aggregate_tables", []):
                show(table_summary)
            print("timings:")
            show(assemble_result.get("timings", {}))



=== summaries: SLIDE-0329_crop_2048 ===
aggregate tables:
timings:
{}

=== summaries: SLIDE-0329_crop_2048_2 ===
aggregate tables:
{
  "name": "agg_cell_labels",
  "labels_layer": "cell_labels",
  "row_count": 4895,
  "feature_count": 24,
  "features": [
    "R1_CY3_AF",
    "R1_CY5_AF",
    "R1_CY7_AF",
    "R1_FITC_AF",
    "R1_BIT2_RS0584_CY3B",
    "R1_BIT3_RS0015_CY5",
    "R1_BIT4_RS0083_750",
    "R1_DAPI",
    "R1_BIT1_RS0996_488",
    "R2_BIT6_RS0639_CY3B",
    "R2_BIT7_RS0109_CY5",
    "R2_BIT8_RS0255_750",
    "R2_DAPI",
    "R2_BIT5_RS1047_488",
    "R3_BIT10_RS0763_CY3B",
    "R3_BIT11_RS1312_CY5",
    "R3_BIT12_RS0237_750",
    "R3_DAPI",
    "R3_BIT9_RS0805_488",
    "R4_P19_POLYRAT",
    "R4_ANXA10_647",
    "R4_LGALS4_750",
    "R4_DAPI",
    "R4_GFP_POLY_AF488"
  ]
}
{
  "name": "agg_nuclear_labels",
  "labels_layer": "nuclear_labels",
  "row_count": 4339,
  "feature_count": 24,
  "features": [
    "R1_CY3_AF",
    "R1_CY5_AF",
    "R1_CY7_AF",
    "R1_FITC_AF",
    

In [15]:
from spatialdata import SpatialData, read_zarr

store = runtime_artifact_paths[SLIDE_ID]["store_path"]
sdata = read_zarr(store)


no parent found for <ome_zarr.reader.Label object at 0x739ed535ba40>: None
no parent found for <ome_zarr.reader.Label object at 0x739ed05ae960>: None


In [16]:
sdata

SpatialData object, with associated Zarr store: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_spatialdata.sdata.zarr
├── Images
│     └── 'full_image': DataTree[cyx] (24, 2048, 2048), (24, 1024, 1024), (24, 512, 512), (24, 256, 256), (24, 128, 128), (24, 64, 64), (24, 32, 32), (24, 16, 16)
├── Labels
│     ├── 'cell_labels': DataArray[yx] (2048, 2048)
│     └── 'nuclear_labels': DataArray[yx] (2048, 2048)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (4895, 4) (2D shapes)
│     └── 'nuclear_boundaries': GeoDataFrame shape: (4339, 4) (2D shapes)
└── Tables
      ├── 'agg_cell_labels': AnnData (4895, 24)
      ├── 'agg_nuclear_labels': AnnData (4339, 24)
      └── 'nimbus_table': AnnData (4895, 4)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images), cell_labels (Labels), nuclear_labels (Labels), cell_boundaries (Shapes), nuclear_boundaries (Shapes)

In [ ]:
from spatialdata import read_zarr
import napari
from napari_spatialdata import Interactive

store = runtime_artifact_paths[SLIDE_ID]["store_path"]
sdata = read_zarr(store)


In [ ]:
from napari_spatialdata import Interactive

interactive = Interactive(sdata)
interactive.run()